<span style="font-weight:bold; font-size: 3rem; color:#333;">- Part 02: Feature pipeline for Train Delay Data (Two-Stage Model)</span>

## 🗒️ Overview

This notebook downloads new data and ingests it into hopsworks feature groups.

It performs the following steps:

1. 


### 📝 Imports

In [29]:
# top of notebook
from features import (
    build_features,
    add_calendar_features,
    add_train_lag_features,
    add_station_network_state_features,
    detect_trigger_time,
    add_reactive_early_dynamics,
    add_weather_rolling_features_if_present,
    add_station_delay_features,
    ensure_event_time,
    add_cause_flags,
    add_station_congestion_features,
    build_duration_baseline,
)


In [30]:
import os
import pandas as pd
import requests
import hopsworks_utils
import datetime as dt
from dotenv import load_dotenv
import hopsworks
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd


# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

load_dotenv()

True

## 📡 Connect to Hopsworks Feature Store

In [31]:
#### it need to get fixed

try:
    project = hopsworks_utils.HopsworksInterface()
    print("Hopsworks login OK")
except Exception as e:
    project = None
    print("Hopsworks not configured / login failed (OK). Proceeding without it.")
    print("Reason:", repr(e))

#train_feature_df = project.get("train_stop_events_labeled")

#uncoment the below line when weather features are stored
#weather_df = project.get("weather_features") 


HOPSWORKS_API_KEY exists: True
HOPSWORKS_API_KEY length: 81
2026-01-11 18:13:22,889 INFO: Closing external client and cleaning up certificates.
2026-01-11 18:13:22,958 INFO: Connection closed.
2026-01-11 18:13:22,989 INFO: Initializing external client
2026-01-11 18:13:22,991 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-01-11 18:13:24,509 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/2182
Hopsworks login OK


In [32]:
# Paths
FG_NAME = "train_stop_events_labeled"



OUT_DIR = "data/feature_pipeline_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# Label settings (must match Part 01)
HORIZON_MIN = int(os.getenv("HORIZON_MIN", "60"))
DELAY_THRESHOLD_MIN = int(os.getenv("DELAY_THRESHOLD_MIN", "10"))

# Rolling windows for network-state features
ROLL_WINDOWS_MIN = [1440, 1440*2]   # minutes
WEATHER_ROLL_WINDOWS_H = [3, 6]  # hours (only used if weather columns exist)

# Split ratios (time-based, by unique dates)
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15

assert abs(TRAIN_FRAC + VAL_FRAC + TEST_FRAC - 1.0) < 1e-9


In [33]:
df = project.get(FG_NAME)

print("Loaded:", FG_NAME)
print("Shape:", df.shape)
display(df.head())


Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (6.63s) 
Loaded: train_stop_events_labeled
Shape: (78045, 31)


,activityid,activitytype,train_id,event_time,scheduled_time,estimated_time,actual_time,observed_time,station_code,delay_min,...,final_delay_min,additional_delay_min,y_delay_within_horizon,temperature_2m,precipitation,rain,snowfall,windspeed_10m,weather_time,source
0,1500adde-075d-66fb-08de-437353752405,Avgang,7879,2026-01-08 23:07:00,2026-01-08 23:07:00,NaT,2026-01-08 23:07:00,2026-01-08 23:07:00,Arns,0.0,...,0.0,0.0,0,-2.0,0.0,0.0,0.0,8.4,2026-01-08 23:00:00,archive
1,1500adde-075d-66fb-08de-43732847e403,Ankomst,2983,2026-01-08 23:09:00,2026-01-08 23:09:00,NaT,2026-01-08 23:07:00,2026-01-08 23:07:00,Upv,-2.0,...,0.0,2.0,0,-2.0,0.0,0.0,0.0,8.4,2026-01-08 23:00:00,archive
2,1500adde-075d-66fb-08de-43732847e56c,Avgang,2983,2026-01-08 23:13:00,2026-01-08 23:13:00,NaT,2026-01-08 23:13:00,2026-01-08 23:13:00,R,0.0,...,0.0,0.0,0,-2.0,0.0,0.0,0.0,8.4,2026-01-08 23:00:00,archive
3,1500adde-075d-66fb-08de-43730fbb830d,Ankomst,2584,2026-01-08 23:20:00,2026-01-08 23:20:00,NaT,2026-01-08 23:19:00,2026-01-08 23:19:00,Gdv,-1.0,...,0.0,1.0,0,-2.0,0.0,0.0,0.0,8.4,2026-01-08 23:00:00,archive
4,1500adde-075d-66fb-08de-43732839343a,Ankomst,2982,2026-01-08 23:20:00,2026-01-08 23:20:00,NaT,2026-01-08 23:19:00,2026-01-08 23:19:00,Rön,-1.0,...,0.0,1.0,0,-2.0,0.0,0.0,0.0,8.4,2026-01-08 23:00:00,archive


In [34]:
# Ensure required columns exist
required_cols = ["event_time", "station_code", "train_id", "delay_min",
                 "y_delay_within_horizon", "final_delay_min", "additional_delay_min"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in canonical dataset: {missing}")

df["event_time"] = pd.to_datetime(df["event_time"], errors="coerce")
df = df.dropna(subset=["event_time", "station_code", "train_id"]).copy()

# Ensure a train-run key exists
if "train_run_id" not in df.columns:
    df["date"] = df["event_time"].dt.date
    df["train_run_id"] = df["train_id"].astype(str) + "_" + df["date"].astype(str)

# Sort for point-in-time computations
df = df.sort_values(["event_time", "station_code", "train_run_id"]).reset_index(drop=True)

# Normalize reason_code
if "reason_code" in df.columns:
    df["reason_code"] = df["reason_code"].astype("string")
else:
    df["reason_code"] = pd.Series([pd.NA]*len(df), dtype="string")

# --- RENAME STEP ADDED HERE ---
weather_rename_map = {
    "temperature_2m": "weather_temperature_2m",
    "precipitation": "weather_precipitation",
    "rain": "weather_rain",
    "snowfall": "weather_snowfall",
    "windspeed_10m": "weather_windspeed_10m"
}
existing_rename = {k: v for k, v in weather_rename_map.items() if k in df.columns}
if existing_rename:
    print(f"Renaming weather columns: {list(existing_rename.keys())}")
    df = df.rename(columns=existing_rename)
# ------------------------------

print("After cleaning:", df.shape)

Renaming weather columns: ['temperature_2m', 'precipitation', 'rain', 'snowfall', 'windspeed_10m']
After cleaning: (78045, 32)


### Feature engineering (no leakage)

In [35]:
print("⚡️ Engineering features using build_features()...")
df_feat = build_features(df)
print("✅ Feature engineering complete.", df_feat.shape)


⚡️ Engineering features using build_features()...
✅ Feature engineering complete. (78045, 65)


In [36]:
print(df.info())
print([c for c in ["deviation", "observed_time", "estimated_time", "event_time"] if c in df_feat.columns])
df_check = df_feat[
    ["event_time", "deviation", "observed_time"]
].dropna(subset=["deviation", "observed_time"])

# Ensure both datetime columns have compatible timezones
df_check["observed_time"] = pd.to_datetime(df_check["observed_time"], utc=True).dt.tz_convert('Europe/Stockholm')
df_check["event_time"] = pd.to_datetime(df_check["event_time"], utc=True).dt.tz_convert('Europe/Stockholm')

df_check["delta_minutes"] = (
    df_check["event_time"] - df_check["observed_time"]
).dt.total_seconds() / 60

df_check["delta_minutes"].describe()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78045 entries, 0 to 78044
Data columns (total 32 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   activityid              78045 non-null  object        
 1   activitytype            78045 non-null  object        
 2   train_id                78045 non-null  object        
 3   event_time              78045 non-null  datetime64[us]
 4   scheduled_time          78045 non-null  datetime64[us]
 5   estimated_time          11463 non-null  datetime64[us]
 6   actual_time             51838 non-null  datetime64[us]
 7   observed_time           51956 non-null  datetime64[us]
 8   station_code            78045 non-null  object        
 9   delay_min               51956 non-null  float64       
 10  is_canceled             78045 non-null  bool          
 11  deleted                 78045 non-null  bool          
 12  informationowner        78045 non-null  object

count    13191.000000
mean       -64.980259
std         14.108116
min       -449.000000
25%        -63.000000
50%        -60.000000
75%        -60.000000
max         41.894117
Name: delta_minutes, dtype: float64

In [37]:
df_check2 = df_feat[
    ["event_time", "deviation", "estimated_time"]
].dropna(subset=["deviation", "estimated_time"])


# Ensure both datetime columns have compatible timezones
df_check2["estimated_time"] = pd.to_datetime(df_check2["estimated_time"], utc=True).dt.tz_convert('Europe/Stockholm')
df_check2["event_time"] = pd.to_datetime(df_check2["event_time"], utc=True).dt.tz_convert('Europe/Stockholm')

df_check2["delta_minutes_est"] = (
    df_check2["event_time"] - df_check2["estimated_time"]
).dt.total_seconds() / 60

df_check2["delta_minutes_est"].describe()


count    3340.000000
mean      -75.504491
std        19.366478
min      -236.000000
25%       -80.000000
50%       -70.000000
75%       -63.000000
max       -61.000000
Name: delta_minutes_est, dtype: float64

In [38]:
# Apply feature engineering
"""
print("⚡️ Engineering features using 'features.py'...")

df_feat = df.copy()

# Ensure event_time is timezone-aware
df_feat = ensure_event_time(df_feat)

# Add cause flags based on reason columns
df_feat = add_cause_flags(df_feat)

# Add station congestion features (lag + rolling counts)
df_feat = add_station_congestion_features(df_feat, windows_min=ROLL_WINDOWS_MIN)

# 1. Calendar
df_feat = add_calendar_features(df_feat)

# 2. Lag features
df_feat = add_train_lag_features(df_feat)

# 3. Network State
df_feat = add_station_network_state_features(df_feat, windows_min=ROLL_WINDOWS_MIN)

# 4. Trigger Detection
df_feat = detect_trigger_time(df_feat)

# 5. Reactive Dynamics
#df_feat = add_reactive_early_dynamics(df_feat)

# 6. Weather
df_feat = add_weather_rolling_features_if_present(df_feat, windows_h=WEATHER_ROLL_WINDOWS_H)

# 7. Station delay features
df_feat = add_station_delay_features(df_feat)

print("✅ Feature engineering complete.")
print("Feature table shape:", df_feat.shape)
df_feat.sort_values(by=["delay_min"], ascending=False, inplace=True)

print(df_feat.info(verbose=True))
display(df_feat.head())
"""
df_feat.sort_values(by=["event_time"], ascending=False, inplace=True)
display(df_feat.head())

,event_time,activityid,activitytype,train_id,scheduled_time,estimated_time,actual_time,observed_time,station_code,delay_min,...,weather_precipitation_rollmean_3h,weather_rain_rollmean_3h,weather_snowfall_rollmean_3h,weather_windspeed_10m_rollmean_3h,weather_temperature_2m_rollmean_6h,weather_precipitation_rollmean_6h,weather_rain_rollmean_6h,weather_snowfall_rollmean_6h,weather_windspeed_10m_rollmean_6h,station_avg_delay
1920,2026-01-12 17:11:00+01:00,1500adde-075d-66fb-08de-45d20fbef252,Avgang,7793,2026-01-12 17:11:00,NaT,NaT,NaT,Arnn,NaN,...,0.003333,0.0,0.002333,9.806667,-3.478333,0.001667,0.0,0.001167,9.291667,0.511312
3225,2026-01-12 17:11:00+01:00,1500adde-075d-66fb-08de-45d20fbbdfa3,Avgang,7792,2026-01-12 17:11:00,NaT,NaT,NaT,Arns,NaN,...,0.003333,0.0,0.002333,9.806667,-3.480000,0.001667,0.0,0.001167,9.286667,0.951412
72712,2026-01-12 17:11:00+01:00,1500adde-075d-66fb-08de-45d1d5071c45,Ankomst,2757,2026-01-12 17:11:00,NaT,NaT,NaT,Äs,NaN,...,0.006736,0.0,0.004715,9.777202,-3.472141,0.003812,0.0,0.002669,9.336657,2.002735
17966,2026-01-12 17:11:00+01:00,1500adde-075d-66fb-08de-45d17832541a,Ankomst,12855,2026-01-12 17:11:00,NaT,NaT,NaT,Hnd,NaN,...,0.003571,0.0,0.002500,9.804762,-3.479114,0.001899,0.0,0.001329,9.321519,0.504865
8097,2026-01-12 17:11:00+01:00,1500adde-075d-66fb-08de-45d1a67040d3,Avgang,20958,2026-01-12 17:11:00,NaT,NaT,NaT,Cst,NaN,...,0.003200,0.0,0.002240,9.821600,-3.481982,0.001802,0.0,0.001261,9.359910,7.043466


In [39]:
import os
import pandas as pd
import numpy as np
import json
import datetime as dt

# --- 0. PRE-REQUISITE: DEFINE FEATURE COLUMNS ---
# Define which columns are ID/Target/Future and should be excluded
ALL_KEYS = ["ActivityId", "train_id", "lag_y_delay", "InformationOwner", "scheduled_time", "estimated_time", "actual_time", "observed_time", "reason_code", "reason_text", "reason_desc", "OperationalTrainNumber", "station_code", "event_time", "event_date", "train_run_id"]
TARGETS = ["y_delay_within_horizon", "final_delay_min", "additional_delay_min"]

# Columns to exclude from Predictive Model (future leakage or reactive-only)
PRED_EXCLUDE = set(TARGETS + ["final_delay_min", "additional_delay_min", 
                              "trigger_time", "min_since_trigger", "delay_at_trigger", 
                              "delay_slope_since_trigger", "is_first10m_after_trigger"])

# Columns to exclude from Reactive Model (just the predictive target)
REACT_EXCLUDE = set(["y_delay_within_horizon"])

# Calculate the lists of columns dynamically from df_feat
pred_cols = [c for c in df_feat.columns if c not in ALL_KEYS and c not in PRED_EXCLUDE]
react_cols = [c for c in df_feat.columns if c not in ALL_KEYS and c not in TARGETS and c not in REACT_EXCLUDE]

print(f"Features detected: {len(pred_cols)} Predictive, {len(react_cols)} Reactive")


# --- 1. SPLIT LOGIC ---
df_feat["event_date"] = df_feat["event_time"].dt.date
dates = sorted(df_feat["event_date"].unique())
n = len(dates)

if n >= 3:
    n_val = max(1, int(np.floor(n * 0.15)))  
    n_test = max(1, int(np.floor(n * 0.15))) 
    n_train = n - n_val - n_test
else:
    n_train = n
    n_val = 0
    n_test = 0

print(f"Split dates: Total={n}d, Train={n_train}d, Val={n_val}d, Test={n_test}d")

train_dates = set(dates[:n_train])
val_dates   = set(dates[n_train:n_train+n_val])
test_dates  = set(dates[n_train+n_val:])

print(f"Refined Split: Train={len(train_dates)}d, Val={len(val_dates)}d, Test={len(test_dates)}d")



# --- 2. RE-SLICE DATAFRAMES ---
df_train = df_feat[df_feat["event_date"].isin(train_dates)].copy()
df_val   = df_feat[df_feat["event_date"].isin(val_dates)].copy()
df_test  = df_feat[df_feat["event_date"].isin(test_dates)].copy()


# Reactive Slices (Filter for delay >= threshold)
df_react_train = df_train[df_train["delay_min"] >= DELAY_THRESHOLD_MIN].copy()
df_react_val   = df_val[df_val["delay_min"] >= DELAY_THRESHOLD_MIN].copy()
#df_react_test  = df_test[df_test["delay_min"] >= DELAY_THRESHOLD_MIN].copy()
df_react_test = df_test.copy()
# --- 3. REGENERATE X AND y ---
# Predictive Targets
X_pred_train = df_train[pred_cols]
y_pred_train = df_train["y_delay_within_horizon"]

X_pred_val   = df_val[pred_cols]
y_pred_val   = df_val["y_delay_within_horizon"]

X_pred_test  = df_test[pred_cols]
y_pred_test  = df_test["y_delay_within_horizon"]

# Reactive Targets
X_react_train = df_react_train[react_cols]
y_react_train = df_react_train["additional_delay_min"]

X_react_val   = df_react_val[react_cols]
y_react_val   = df_react_val["additional_delay_min"]

X_react_test  = df_react_test[react_cols]
y_react_test  = df_react_test["additional_delay_min"]

# --- 4. PACK AND SAVE ---
# Keys to keep in the final output for joining
SAVE_KEYS = ["train_run_id", "event_time", "station_code", "train_id"]

pred_train_path = "pred_train"
pred_val_path   = "pred_val"
pred_test_path  = "pred_test"

react_train_path = "react_train"
react_val_path   = "react_val"
react_test_path  = "react_test"
"""
def pack(df_split, X, y, task: str) -> pd.DataFrame:
    if df_split.empty:
        return pd.DataFrame()
        
    actual_keys = [k for k in SAVE_KEYS if k in df_split.columns]
    packed = df_split[actual_keys].copy().reset_index(drop=True)
    
    if isinstance(X, pd.DataFrame):
        X = X.reset_index(drop=True)
        packed = pd.concat([packed, X], axis=1)
    else:
        packed = packed.join(pd.DataFrame(X, columns=pred_cols if task=="pred" else react_cols))
        
    # Correctly name the target column
    if task == "pred":
        packed["y_delay_within_horizon"] = y.values
    elif task == "react":
        packed["additional_delay_min"] = y.values
    else:
        packed[f"y_{task}"] = y.values
    
    return packed
"""
def pack(df_split, X, y, task: str) -> pd.DataFrame:
    """
    Always returns a dataframe with the correct schema (even if df_split is empty),
    so parquet files reload with columns instead of (0,0).
    """
    if task == "pred":
        feat_cols = pred_cols
        target_col = "y_delay_within_horizon"
    elif task == "react":
        feat_cols = react_cols
        target_col = "additional_delay_min"
    else:
        raise ValueError(f"Unknown task={task}")

    # Keys to keep
    actual_keys = [k for k in SAVE_KEYS if (not df_split.empty and k in df_split.columns)]
    if df_split.empty:
        # Create empty keys frame with stable schema
        packed = pd.DataFrame({k: pd.Series(dtype="object") for k in SAVE_KEYS})
        packed = packed[[k for k in SAVE_KEYS]]  # keep order
    else:
        packed = df_split[actual_keys].copy().reset_index(drop=True)

    # Attach X with correct columns (even if empty)
    if isinstance(X, pd.DataFrame):
        X2 = X.copy()
        for c in feat_cols:
            if c not in X2.columns:
                X2[c] = pd.Series(dtype="float64")
        X2 = X2[feat_cols].reset_index(drop=True)
    else:
        X2 = pd.DataFrame(X, columns=feat_cols)

    packed = pd.concat([packed.reset_index(drop=True), X2.reset_index(drop=True)], axis=1)

    # Attach target with correct column name (even if empty)
    if y is None or len(y) == 0:
        packed[target_col] = pd.Series(dtype="float64")
    else:
        packed[target_col] = pd.Series(y.values)

    return packed

def save_fg(df, path):
    if not df.empty:
        #print(df.info())
        #display(df.head())
        keys = ["train_run_id", "activityid"]
        print(df[keys].info())
        

        # assure all elements in df are unique on keys

        print(f"Checking uniqueness on keys: {keys}")
        n_total = len(df)
        n_unique = len(df.drop_duplicates(subset=keys))
        if n_total != n_unique:
            print(f"⚠️ Warning: DataFrame not unique on keys {keys}: total={n_total}, unique={n_unique}")
            raise ValueError("DataFrame uniqueness check failed.")

        project.push(df, path, keys)
        #df.to_parquet(path, index=False)
        print(f"✅ Saved: {path} ({len(df)} rows)")
    else:
        print(f"⚠️ Empty split, saving schema only: {path}")
        project.push(df, path)

        #df.to_parquet(path, index=False)

# Save
save_fg(pack(df_train, X_pred_train, y_pred_train, "pred"), pred_train_path)
save_fg(pack(df_val,   X_pred_val,   y_pred_val,   "pred"), pred_val_path)
save_fg(pack(df_test,  X_pred_test,  y_pred_test,  "pred"), pred_test_path)

save_fg(pack(df_react_train, X_react_train, y_react_train, "react"), react_train_path)
save_fg(pack(df_react_val,   X_react_val,   y_react_val,   "react"), react_val_path)
save_fg(pack(df_react_test,  X_react_test,  y_react_test,  "react"), react_test_path)

# Metadata (Includes feature names for Part 4!)
metadata = {
    "created_utc": dt.datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "splits": {
        "train_dates": [str(d) for d in sorted(train_dates)],
        "val_dates": [str(d) for d in sorted(val_dates)],
        "test_dates": [str(d) for d in sorted(test_dates)],
    },
    "pred_feature_columns": pred_cols,   # <--- Critical for Part 4
    "react_feature_columns": react_cols  # <--- Critical for Part 4
}
meta_path = os.path.join(OUT_DIR, "feature_metadata.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"✅ Metadata saved to {meta_path}")

Features detected: 50 Predictive, 52 Reactive
Split dates: Total=5d, Train=3d, Val=1d, Test=1d
Refined Split: Train=3d, Val=1d, Test=1d
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42426 entries, 0 to 42425
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   train_run_id  42426 non-null  object
 1   activityid    42426 non-null  object
dtypes: object(2)
memory usage: 663.0+ KB
None
Checking uniqueness on keys: ['train_run_id', 'activityid']
2026-01-11 18:14:18,804 INFO: Computing insert statistics
Inserted historical data into feature group "train_delay_features"
✅ Saved: pred_train (42426 rows)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17965 entries, 0 to 17964
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   train_run_id  17965 non-null  object
 1   activityid    17965 non-null  object
dtypes: object(2)
memory usage: 280.8+ KB
None


In [40]:
print("df_test rows:", len(df_test))
print("df_test delay_min stats:")
print(df_test["delay_min"].describe(include="all"))

print("DELAY_THRESHOLD_MIN:", DELAY_THRESHOLD_MIN)
print("Rows in test >= threshold:", (df_test["delay_min"] >= DELAY_THRESHOLD_MIN).sum())
print("NaNs in test delay_min:", df_test["delay_min"].isna().sum())

print("test_dates:", sorted(test_dates)[:5], "...", sorted(test_dates)[-5:])
print("test day counts:")
print(df_test.groupby("event_date").size())
print("test day counts (>=threshold):")
print(df_test[df_test["delay_min"] >= DELAY_THRESHOLD_MIN].groupby("event_date").size())


df_test rows: 17654
df_test delay_min stats:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: delay_min, dtype: float64
DELAY_THRESHOLD_MIN: 10
Rows in test >= threshold: 0
NaNs in test delay_min: 17654
test_dates: [datetime.date(2026, 1, 12)] ... [datetime.date(2026, 1, 12)]
test day counts:
event_date
2026-01-12    17654
dtype: int64
test day counts (>=threshold):
Series([], dtype: int64)


In [41]:
#List columns in training and test data
#print("Predictive feature columns:", pred_cols)

display(df_train[pred_cols].head())
#print(df_train.columns.tolist())

,activityid,activitytype,delay_min,is_canceled,deleted,informationowner,deviation,fromlocation,tolocation,trackatlocation,...,weather_precipitation_rollmean_3h,weather_rain_rollmean_3h,weather_snowfall_rollmean_3h,weather_windspeed_10m_rollmean_3h,weather_temperature_2m_rollmean_6h,weather_precipitation_rollmean_6h,weather_rain_rollmean_6h,weather_snowfall_rollmean_6h,weather_windspeed_10m_rollmean_6h,station_avg_delay
60401,1500adde-075d-66fb-08de-45074c91aaeb,Avgang,0.0,False,False,SL,Kort tåg,Söc,"Sci,Mr",2,...,0.0,0.0,0.0,25.295556,-3.687097,0.0,0.0,0.0,25.639785,3.113983
39108,1500adde-075d-66fb-08de-45074cac38f7,Ankomst,-1.0,False,False,SL,None,Mr,Äs,2,...,0.0,0.0,0.0,25.156522,-3.670213,0.0,0.0,0.0,25.577660,2.835623
60400,1500adde-075d-66fb-08de-45074c91aaea,Ankomst,-1.0,False,False,SL,None,Söc,Mr,2,...,0.0,0.0,0.0,25.363636,-3.691304,0.0,0.0,0.0,25.676087,3.113983
4988,1500adde-075d-66fb-08de-450736fa2815,Avgang,0.0,False,False,SL,None,Bål,"Sci,Nyc",2,...,0.0,0.0,0.0,25.250000,-3.700000,0.0,0.0,0.0,25.587500,0.965986
4989,1500adde-075d-66fb-08de-450736fa2814,Ankomst,-1.0,False,False,SL,None,Bål,Nyc,2,...,0.0,0.0,0.0,25.132000,-3.691837,0.0,0.0,0.0,25.520408,0.965986


In [42]:
# Verify weather columns are in the list
weather_cols_saved = [c for c in pred_cols if "weather" in c]
print(f"n📊 Verification: {len(weather_cols_saved)} weather features included in predictive model.")
if len(weather_cols_saved) > 0:
    print("Example features:", weather_cols_saved[:3])
else:
    print("⚠️ WARNING: No weather features detected in the final list!")


n📊 Verification: 17 weather features included in predictive model.
Example features: ['weather_temperature_2m', 'weather_precipitation', 'weather_rain']


In [43]:
print("done")

done


### ✂️ Time-based train/val/test split (no leakage)

### 🧩 Build predictive vs reactive feature matrices

### 💾 Save outputs + feature metadata